Paper: https://aclanthology.org/2021.emnlp-main.612.pdf

Github: https://github.com/nattaptiy/qe_disentangled

Eval data of paper: 
- Task 3 Document-Level QA: https://github.com/facebookresearch/mlqe, https://www.statmt.org/wmt20/quality-estimation-task.html
- STS17: https://public.ukp.informatik.tu-darmstadt.de/reimers/sentence-transformers/datasets/STS2017-extended.zip

# Encoder

In [1]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('sentence-transformers/LaBSE')

sentence = 'Find embedding size'
embedding_dim = encoder.encode(sentence).shape[0]
print(embedding_dim)

768


In [2]:
import gc
import torch

def free(encoder: SentenceTransformer):
    try:
        encoder.to('cpu')
    except:
        pass

    del encoder

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # torch.cuda.synchronize()   # optional
        gc.collect()
        torch.cuda.empty_cache()

# Train, Val Split

In [3]:
# # Run ONCE
# import pandas as pd
# import glob
# import os

# VAL_SIZE = 0.1
# PATH = '../data/Tatoeba'
# TRAIN_SUFFIX = '_Train'
# VAL_SUFFIX   = '_Val'

# os.makedirs(f"{PATH}{VAL_SUFFIX}", exist_ok=True)
# os.makedirs(f"{PATH}{TRAIN_SUFFIX}", exist_ok=True)

# all_tsv = glob.glob(f'{PATH}/*.tsv')

# for tsv_path in all_tsv:
#     filename = os.path.basename(tsv_path)
#     name_only = os.path.splitext(filename)[0]

#     try:
#         df = pd.read_csv(
#             tsv_path,
#             sep='\t',
#             header=None,
#             names=['src_id', 'src', 'tar_id', 'tar'],
#             on_bad_lines='skip',      
#             dtype=str
#         )
        
#         if len(df) == 0:
#             print(f"Empty file: {tsv_path}")
#             continue
            
#         # Shuffle và split
#         val_df = df.sample(frac=VAL_SIZE, random_state=42)          # fixed seed → reproducible
#         train_df = df.drop(val_df.index)                            # cách an toàn nhất
        
#         # Save to folders
#         val_path  = f"{PATH}{VAL_SUFFIX}/{filename}"
#         train_path = f"{PATH}{TRAIN_SUFFIX}/{filename}"

#         val_df.to_csv(val_path, sep='\t', header=False, index=False)
#         train_df.to_csv(train_path, sep='\t', header=False, index=False)
        
#         print(f"Done | train: {len(train_df):,} | val: {len(val_df):,}")
        
#     except Exception as e:
#         print(f"Error Processing {tsv_path}: {str(e)}")
#         continue

# Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import random
import glob

class SingleTatoebaDataset(Dataset):
    """
    PyTorch Dataset for the Tatoeba parallel corpus (single language pair).

    Each sample returns 4 elements:
        (a) src_id of a synonym pair
        (b) tar_id of a synonym pair
        (c) src_id of a random NON-synonym pair
        (d) tar_id of a random NON-synonym pair

    Use src_lookup / tar_lookup to decode id → text when needed.
    Call shuffle() at the beginning of each epoch to regenerate random pairs.
    """

    def __init__(self, tsv_path: str, shuffle=True, encoder=None) -> None:
        self.path = tsv_path
        data = pd.read_csv(
            tsv_path, sep="\t", header=None,
            names=["src_id", "src", "tar_id", "tar"],
            on_bad_lines="skip"
        )
        data = data.dropna(subset=["src_id", "src", "tar_id", "tar"])

        self.is_encoded = encoder is not None
        if self.is_encoded:
            src_df = data.drop_duplicates("src_id")[["src_id", "src"]]
            src_embeddings = encoder.encode(
                src_df['src'].to_list(),
                batch_size=64,
                convert_to_tensor=True,
                normalize_embeddings=False,
                show_progress_bar=True,
                device = "cuda" if torch.cuda.is_available() else "cpu"
            )
            self.src_id_to_embedding_index = {src_id: idx for idx, src_id in enumerate(src_df['src_id'].tolist())}
            self.src_embedding_vectors = src_embeddings

            tar_df = data.drop_duplicates("tar_id")[["tar_id", "tar"]]
            tar_embeddings = encoder.encode(
                tar_df['tar'].to_list(),
                batch_size=64,
                convert_to_tensor=True,
                normalize_embeddings=False,
                show_progress_bar=True,
                device = "cuda" if torch.cuda.is_available() else "cpu"
            )
            self.tar_id_to_embedding_index = {tar_id: idx for idx, tar_id in enumerate(tar_df['tar_id'].tolist())}
            self.tar_embedding_vectors = tar_embeddings


        else:
            # id → text mapping, used for decoding during inference or encoding
            self.src_lookup: dict[int, str] = (
                data.drop_duplicates("src_id")
                    .set_index("src_id")["src"]
                    .to_dict()
            )
            self.tar_lookup: dict[int, str] = (
                data.drop_duplicates("tar_id")
                    .set_index("tar_id")["tar"]
                    .to_dict()
            )

        # src_id → set of synonym tar_ids, used for conflict detection
        self.synonym_lookup: dict[int, set[int]] = (
            data.groupby("src_id")["tar_id"].apply(set).to_dict()
        )

        # Ground-truth synonym pairs — order is preserved across epochs
        self.synonym_pairs: list[tuple[int, int]] = list(
            zip(data["src_id"].values, data["tar_id"].values)
        )
        if shuffle:
            random.shuffle(self.synonym_pairs)

        # Pre-computed random pairs; regenerated each epoch via shuffle()
        self.random_pairs: list[tuple[int, int]] = self._build_random_pairs()

    # ------------------------------------------------------------------

    def _build_random_pairs(self, max_attempts: int = 10) -> list[tuple[int, int]]:
        """
        Build a list of random pairs guaranteed to contain no synonym pairs.

        Strategy:
            - Shuffle tar_ids, then resolve conflicts via swapping.
            - If a conflict cannot be resolved by swapping, re-shuffle entirely.

        Args:
            max_attempts: maximum number of re-shuffle attempts before raising.

        Returns:
            List of (src_id, tar_id) where no pair is a known synonym.

        Raises:
            RuntimeError: if conflicts cannot be resolved after max_attempts.
        """
        # src_ids are kept in original order to stay aligned with synonym_pairs
        src_ids = [src_id for src_id, _ in self.synonym_pairs]
        tar_ids = [tar_id for _, tar_id in self.synonym_pairs]
        N = len(src_ids)

        random.shuffle(src_ids)

        for attempt in range(1, max_attempts + 1):
            random.shuffle(tar_ids)

            has_unresolved = False

            for i in range(N):
                synonyms_i = self.synonym_lookup.get(src_ids[i], set())

                # No conflict at position i, skip
                if tar_ids[i] not in synonyms_i:
                    continue

                # Conflict detected — find j to swap with
                is_swapped = False
                for j in range(i + 1, N):
                    synonyms_j = self.synonym_lookup.get(src_ids[j], set())
                    if tar_ids[j] not in synonyms_i and tar_ids[i] not in synonyms_j:
                        tar_ids[i], tar_ids[j] = tar_ids[j], tar_ids[i]
                        is_swapped = True
                        break  # stop after first valid swap

                # No valid j found — trigger a full re-shuffle
                if not is_swapped:
                    has_unresolved = True
                    break

            if not has_unresolved:
                return list(zip(src_ids, tar_ids))

        raise RuntimeError(
            f"Could not resolve all synonym conflicts after {max_attempts} attempts. "
            "Dataset may be too small or synonym density too high."
        )

    # ------------------------------------------------------------------

    def shuffle(self) -> None:
        """
        Regenerate random pairs with a new random order.
        Should be called at the start of each epoch to prevent the model
        from memorizing fixed negative patterns.
        """
        random.shuffle(self.synonym_pairs)
        self.random_pairs = self._build_random_pairs()

    def __len__(self) -> int:
        return len(self.synonym_pairs)

    def __getitem__(self, index: int) -> tuple[int, int, int, int]:
        """
        Returns:
            (a_src_id, b_tar_id, c_src_id, d_tar_id) where
            (a, b) is a synonym pair and (c, d) is a non-synonym pair.
        """
        synonym_src_id, synonym_tar_id = self.synonym_pairs[index]
        random_src_id, random_tar_id = self.random_pairs[index]

        if not self.is_encoded:
            return self.src_lookup[synonym_src_id], self.tar_lookup[synonym_tar_id], self.src_lookup[random_src_id], self.tar_lookup[random_tar_id]
        
        synonym_src_embedding_idx = self.src_id_to_embedding_index[synonym_src_id]
        synonym_tar_embedding_idx = self.tar_id_to_embedding_index[synonym_tar_id]
        random_src_embedding_idx  = self.src_id_to_embedding_index[random_src_id]
        random_tar_embedding_idx  = self.tar_id_to_embedding_index[random_tar_id]

        return (self.src_embedding_vectors[synonym_src_embedding_idx], 
                self.tar_embedding_vectors[synonym_tar_embedding_idx],
                self.src_embedding_vectors[random_src_embedding_idx], 
                self.tar_embedding_vectors[random_tar_embedding_idx])
    

class TatoebaDataset(Dataset):
    def __init__(self, folder_path, shuffle=True, encoder=None):
        all_tsv_paths = glob.glob(f'{folder_path}/*tsv')
        self.single_datasets = [SingleTatoebaDataset(tsv_path=tsv_path, shuffle=shuffle, encoder=encoder) for tsv_path in all_tsv_paths]
        self.flat_index = [(dataset_index, data_index) 
                           for dataset_index, single_dataset in  enumerate(self.single_datasets) 
                           for data_index in range(len(single_dataset))]
        random.shuffle(self.flat_index)

    def shuffle(self):
        for dataset in self.single_datasets:
            dataset.shuffle()

        self.flat_index = [(dataset_index, data_index) 
                           for dataset_index, single_dataset in  enumerate(self.single_datasets) 
                           for data_index in range(len(single_dataset))]
        
        random.shuffle(self.flat_index)

    def __len__(self):
        return len(self.flat_index)

    def __getitem__(self, index):
        dataset_index, data_index = self.flat_index[index]

        # Unpack 4 embeddings from SingleTatoebaDataset
        src, tar, rand_src, rand_tar = self.single_datasets[dataset_index][data_index]

        # src is always English = 0
        src_lang_id = 0

        # trg_lang is dataset_index (file alphabetical)
        # Arabic=1, Dutch=2, French=3, German=4, Italian=5, Spanish=6, Turkish=7
        tar_lang_id = dataset_index + 1

        return src, tar, rand_src, rand_tar, src_lang_id, tar_lang_id

In [ ]:
from dataset import TatoebaDataset

train_dataset = TatoebaDataset('../data/Tatoeba_Train', encoder=encoder)
val_dataset = TatoebaDataset('../data/Tatoeba_Val', encoder=encoder) 

Batches:   0%|          | 0/618 [00:00<?, ?it/s]

Batches:   0%|          | 0/654 [00:00<?, ?it/s]

Batches:   0%|          | 0/2021 [00:00<?, ?it/s]

Batches:   0%|          | 0/2082 [00:00<?, ?it/s]

Batches:   0%|          | 0/4709 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1087 [00:00<?, ?it/s]

Batches:   0%|          | 0/428 [00:00<?, ?it/s]

Batches:   0%|          | 0/434 [00:00<?, ?it/s]

Batches:   0%|          | 0/1110 [00:00<?, ?it/s]

Batches:   0%|          | 0/1105 [00:00<?, ?it/s]

In [5]:
free(encoder)

In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE)

# Model

In [ ]:
import torch
import torch.nn as nn

class DREAMModel(nn.Module):
    def __init__(self, embedding_size, num_languages):
        super(DREAMModel, self).__init__()
        self.language_encoder = nn.Linear(embedding_size, embedding_size)
        self.meaning_encoder = nn.Linear(embedding_size, embedding_size)
        self.language_identifier = nn.Linear(embedding_size, num_languages)

    def forward(self, sentence_embedding):
        language_embedding = self.language_encoder(sentence_embedding)
        meaning_embedding  = self.meaning_encoder(sentence_embedding)
        language_id = self.language_identifier(language_embedding)
        return language_embedding, meaning_embedding, language_id

In [7]:
from model import DREAMModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DREAMModel(embedding_dim, num_languages=8).to(DEVICE)

# Optimizer

In [8]:
import torch

LEARNING_RATE = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Loss

In [9]:
import torch
import torch.nn.functional as F

def reconstruction_loss(e, eM, eL):
    return ((e - (eM + eL)) ** 2).mean()

def meaning_loss(sM, tM, rand_sM, rand_tM):
    # The meaning of 2 parallel sentences should be close
    parallel_sim = F.cosine_similarity(sM, tM, dim=-1)
    Lx = (1 - parallel_sim).mean()

    # The meaning of 2 random sentences should be far
    source_sim = F.cosine_similarity(sM, rand_sM, dim=-1)
    random_sim = F.cosine_similarity(tM, rand_tM)
    Lm = (torch.clamp(source_sim, min=0) + torch.clamp(random_sim, min=0)).mean()

    return Lx + Lm

def language_loss(sL, tL, rand_sL, rand_tL, sI, tI, src_lang_id, tar_lang_id):
    # Same-language embeddings should be close (được optimize)
    Lm = (2 - F.cosine_similarity(sL, rand_sL, dim=-1) 
             - F.cosine_similarity(tL, rand_tL, dim=-1)).mean()

    # Cross-language embeddings should be far (for reserving the author's code only)
    Lm_cross = torch.clamp(F.cosine_similarity(sL, tL, dim=-1), min=0).mean()

    # Language identification
    Li = F.cross_entropy(sI, src_lang_id) + F.cross_entropy(tI, tar_lang_id)

    return Lm + Li


def total_loss(e_src, e_trg,
               eM_src, eM_trg, eM_rand_src, eM_rand_trg,
               eL_src, eL_trg, eL_rand_src, eL_rand_trg,
               eI_src, eI_trg,
               src_lang_id, tar_lang_id):

    LR = reconstruction_loss(e_src, eM_src, eL_src) \
       + reconstruction_loss(e_trg, eM_trg, eL_trg)

    LM = meaning_loss(eM_src, eM_trg, eM_rand_src, eM_rand_trg)

    LL = language_loss(eL_src, eL_trg, eL_rand_src, eL_rand_trg,
                       eI_src, eI_trg,
                       src_lang_id, tar_lang_id)

    return LR + LM + LL, LR, LM, LL

# Train

In [ ]:
import time

EPOCH = 10

for epoch in range(1, EPOCH + 1):
    t0 = time.time()

    train_dataset.shuffle()

    model.train()
    train_total = train_rec = train_mean = train_lang = 0.0
    for batch in train_loader:
        e_src, e_trg, rand_src, rand_tar, src_lang_id, tar_lang_id = [x.to(DEVICE) for x in batch]

        optimizer.zero_grad()

        eL_src, eM_src, eI_src = model(e_src)
        eL_trg, eM_trg, eI_trg = model(e_trg)
        eL_rand_src, eM_rand_src, _ = model(rand_src)
        eL_rand_trg, eM_rand_trg, _ = model(rand_tar)

        loss, loss_rec, loss_mean, loss_lang = total_loss(e_src, e_trg, eM_src, eM_trg,
                                                        eM_rand_src, eM_rand_trg, eL_src, eL_trg, eL_rand_src, eL_rand_trg, 
                                                        eI_src, eI_trg, src_lang_id, tar_lang_id)

        train_total += loss.item()
        train_rec   += loss_rec.item()
        train_mean  += loss_mean.item()
        train_lang  += loss_lang.item()

        loss.backward()
        optimizer.step()


    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            e_src, e_trg, rand_src, rand_tar, src_lang_id, tar_lang_id = [x.to(DEVICE) for x in batch]

            eL_src, eM_src, eI_src = model(e_src)
            eL_trg, eM_trg, eI_trg = model(e_trg)
            eL_rand_src, eM_rand_src, _ = model(rand_src)
            eL_rand_trg, eM_rand_trg, _ = model(rand_tar)

            val_loss, loss_rec, loss_mean, loss_lang = total_loss(e_src, e_trg, eM_src, eM_trg,
                                                    eM_rand_src, eM_rand_trg, eL_src, eL_trg, eL_rand_src, eL_rand_trg, 
                                                    eI_src, eI_trg,src_lang_id, tar_lang_id)

            val_total += val_loss

    t1 = time.time()
    n = len(train_loader)
    train_total /= n
    train_rec   /= n
    train_mean  /= n
    train_lang  /= n

    m = len(val_loader)
    val_total /= m

    print(f"Epoch {epoch:2d} | train={train_total:.4f} (LR={train_rec:.3f} LM={train_mean:.3f} LL={train_lang:.3f}) | val={val_total:.4f}")

Epoch   1 | train=1.3801 (LR=0.045 LM=0.161 LL=1.174) | val=1.2031
Epoch   2 | train=1.1790 (LR=0.034 LM=0.155 LL=0.990) | val=1.1696
Epoch   3 | train=1.1558 (LR=0.029 LM=0.154 LL=0.973) | val=1.1541
Epoch   4 | train=1.1436 (LR=0.027 LM=0.153 LL=0.964) | val=1.1483
Epoch   5 | train=1.1356 (LR=0.025 LM=0.152 LL=0.958) | val=1.1354
Epoch   6 | train=1.1306 (LR=0.023 LM=0.152 LL=0.955) | val=1.1401
Epoch   7 | train=1.1273 (LR=0.022 LM=0.152 LL=0.953) | val=1.1291
Epoch   8 | train=1.1253 (LR=0.022 LM=0.152 LL=0.952) | val=1.1341
Epoch   9 | train=1.1234 (LR=0.021 LM=0.151 LL=0.951) | val=1.1245
Epoch  10 | train=1.1221 (LR=0.020 LM=0.152 LL=0.950) | val=1.1232
